In [1]:
%run /home/hadoop/agrim_cdp/common_utils/db.ipynb

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
import requests
import psycopg2
from datetime import datetime, timedelta, timezone
from psycopg2.extras import execute_values
import logging

In [3]:
# Load credentials and configs from environment
KNOWLARITY_API_KEY = 'j5Fct1R9Dp9KriJ1r9qEsPayB8nbxwN1K4IEE3h6'       # x-api-key
KNOWLARITY_AUTH_TOKEN = 'effcd553-6e9b-4171-a1a1-444f63897939' # authorization

In [4]:
RDS_HOST, RDS_DBNM, RDS_USER, RDS_PASSWORD, RDS_PORT = get_rds_credentials()

In [5]:
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [6]:
def get_time_window():
    last_run = get_last_run_datetime('knowlarity')
    now = datetime.now(timezone.utc).astimezone()
    end_time = now.strftime("%Y-%m-%d %H:%M:%S%z")
    last_run = (last_run - timedelta(minutes=1))
    return last_run.strftime("%Y-%m-%d %H:%M:%S%z"), end_time

In [7]:
def fetch_call_logs(start_time, end_time):
    url = "https://kpi.knowlarity.com/Basic/v1/account/calllog"
    data = ""
    headers = {
        'channel': "Basic",
        'x-api-key': KNOWLARITY_API_KEY,
        'authorization': KNOWLARITY_AUTH_TOKEN,
        'content-type': "application/json"
    }

    all_logs = []
    offset = 0
    limit = 300

    while True:
        logger.info(f'Fetching Data with Offset - {offset}')
        params = {
            'start_time': start_time,
            'end_time': end_time,
            'limit': limit,
            'offset': offset
        }
        response = requests.get(url, headers=headers, params = params)
        #print(response.json())
        response.raise_for_status()
        resp_json = response.json()

        # Append current page's objects
        logs = resp_json.get('objects', [])
        all_logs.extend(logs)

        meta = resp_json.get('meta', {})
        next_url = meta.get('next')

        offset = offset + limit

        if not next_url:
            logger.info(f'No More Records to Fetch. Breaking here ....')
            break  # No more pages

    return all_logs

In [8]:
def normalize_call_log(raw_logs):
    normalized_logs = []
    for log in raw_logs:
        normalized_logs.append((
            log.get('id'),
            str(log.get('Call_Type')),  # stored as text
            log.get('agent_number'),
            log.get('business_call_type'),
            str(log.get('call_duration')),
            log.get('call_recording'),
            log.get('caller_name'),
            log.get('credits_deducted'),
            log.get('customer_number'),
            log.get('destination'),
            log.get('extension'),
            log.get('knowlarity_number'),
            str(log.get('order_id')),
            log.get('start_time'),
            log.get('timezone_offset'),
            log.get('uuid')
        ))
    return normalized_logs

In [9]:
def insert_into_postgres(logs):
    conn = psycopg2.connect(
        host=RDS_HOST,
        dbname=RDS_DBNM,
        user=RDS_USER,
        password=RDS_PASSWORD,
        port=RDS_PORT
    )
    cur = conn.cursor()

    insert_sql = """
        INSERT INTO cdp_raw_db.knowlarity_call_logs_raw (
            id, call_type, agent_number, business_call_type, call_duration, call_recording,
            caller_name, credits_deducted, customer_number, destination, extension_key,
            knowlarity_number, order_id, start_time, timezone_offset, uuid_key
        )
        VALUES %s
        ON CONFLICT (uuid_key) DO NOTHING
    """
    logger.info('Number of records inserted - ' + str(len(logs)))
    execute_values(cur,insert_sql, logs)
    #cur.execute(insert_sql, logs)

    conn.commit()
    cur.close()
    conn.close()

In [10]:
try:
        start_time, end_time = get_time_window()
        raw_logs = fetch_call_logs(start_time, end_time)
        logger.info(raw_logs)
        normalized_logs = normalize_call_log(raw_logs)
        if normalized_logs:
            insert_into_postgres(normalized_logs)
            update_last_run_datetime('knowlarity',end_time)
        logger.info(f'Process Completed')
except Exception as e:
        logger.info(f'There was an Exception - {e}')

INFO:__main__:Fetching Data with Offset - 0


INFO:__main__:No More Records to Fetch. Breaking here ....


INFO:__main__:[{'customer_number': '+917987620674', 'uuid': '475693ca-cc1a-486d-a361-102a3c7a0e4c', 'agent_number': '+919717894559', 'call_duration': 323, 'business_call_type': 'Phone', 'id': 1, 'order_id': 0, 'destination': '+919717894559', 'Call_Type': 1, 'call_recording': 'https://kservices.knowlarity.com/kstorage/read?uuid=475693ca-cc1a-486d-a361-102a3c7a0e4c_0_1_r.mp3&server_name=BANGALORE&base_dir=recording&user_id=1390233', 'knowlarity_number': '+919513251239', 'start_time': '2025-07-09 15:49:56+05:30', 'credits_deducted': '0.00', 'extension': '', 'caller_name': '', 'timezone_offset': '+0530'}, {'customer_number': '+919199106544', 'uuid': '2157aa97-54f1-41c6-b1ea-902d4943a11e', 'agent_number': 'Customer Missed#+919289589977', 'call_duration': 45, 'business_call_type': 'Phone', 'id': 2, 'order_id': 0, 'destination': '+919289589977', 'Call_Type': 1, 'call_recording': '', 'knowlarity_number': '+918035760523', 'start_time': '2025-07-09 15:49:51+05:30', 'credits_deducted': '0.00', 'e

INFO:__main__:Number of records inserted - 27


INFO:__main__:Process Completed
